## WP007 — Understat Player Data Exploration

Small, deliberately non-modelling task: understand exactly what player-level xG/xA/appearance data `understatapi` (already a project dependency — `get_data.py` uses it for team-level match data) actually exposes, before building anything on top of it for the lineup-quality covariate (WP008+). See `README.md` for the full findings; this notebook is the reproducible source for every claim in it.

In [ ]:
import json
from understatapi import UnderstatClient
from understatapi.exceptions import InvalidPlayer

c = UnderstatClient()
print([m for m in dir(c) if not m.startswith('_')])

### 1. Discovering players: `team.get_player_data(season)`

Season-aggregate totals for every player on a squad, plus the player ID needed for the per-match endpoint below.

In [ ]:
squad = c.team(team='Manchester_City').get_player_data(season='2023')
print(f'{len(squad)} players on the 2023 City squad')
print(json.dumps(squad[0], indent=2))
print()
print('all fields:', list(squad[0].keys()))
print('distinct position codes across the squad:', sorted(set(p['position'] for p in squad)))

### 2. Per-match log: `player.get_match_data()`

This is the actual data source for the lineup covariates. One call per player, **no season parameter** — it returns the player's entire Understat-tracked history in one go (every league, every season), not just the team/season used to discover their ID.

In [ ]:
matches = c.player(player='1228').get_match_data()  # Bruno Fernandes
print(f'{len(matches)} matches total, spanning', min(m['season'] for m in matches), '-', max(m['season'] for m in matches))
print()
print('most recent match (index 0 -- list is newest-first):')
print(json.dumps(matches[0], indent=2))

### 3. Does this reach non-attacking players? Contrast a goalkeeper.

The worry going in: xG/xA-family stats might be ~meaningless for defenders/keepers who rarely shoot or create chances directly. `xGChain`/`xGBuildup` (credit for *any* involvement in a possession that leads to a shot, not just the shot/key-pass itself) turn out to reach even a keeper purely through distribution.

In [ ]:
keeper = [p for p in squad if p['position'] == 'GK'][0]
print(f"{keeper['player_name']} ({keeper['position']}): games={keeper['games']} time={keeper['time']} "
      f"xG={keeper['xG']} xGChain={keeper['xGChain']} xGBuildup={keeper['xGBuildup']}")
print()
print('-> xG is ~0 (keepers essentially never shoot) but xGChain/xGBuildup are real and substantial —')
print('   they DO reach a keeper through pure distribution/buildup involvement, not shot-taking.')

### 4. Join key check against the existing pipeline

The existing `get_understat_data()` pipeline (`match_transformer.py`) drops Understat's own numeric match `id` early (`_clean_data` selects columns without `id`) — so there's no direct id-to-id join available downstream. But team names are a different story:

In [ ]:
import pickle
df_cv = pickle.load(open('/Users/hadiahmed/Documents/projects/football-predictor/work_products/wp001_walkforward_cv_baseline/cv_shared_data.pkl', 'rb'))['df_cv']

existing_names = set(df_cv['team_long'].unique())
player_log_names = {m['h_team'] for m in matches} | {m['a_team'] for m in matches}
overlap_check = {'Manchester City', 'Manchester United'} <= (existing_names | player_log_names)
print('existing df_cv team_long sample:', sorted(existing_names)[:5])
print('player-match-log h_team/a_team sample:', sorted(player_log_names)[:5])
print()
print('Both are sourced from Understat itself, so names should match with zero crosswalk needed —')
print('unlike WP003, which needed a 6-entry crosswalk against football-data.co.uk\'s different naming.')
print('(spot check) "Manchester City" in both:', 'Manchester City' in existing_names and 'Manchester City' in player_log_names)

### 5. Error handling and data-quality notes

- Invalid player ID raises `understatapi.exceptions.InvalidPlayer` cleanly — no silent failure.
- Every numeric field comes back as a **string**, not a float/int — explicit casting needed everywhere downstream (this bit the team-level pipeline too, historically).
- `time` is minutes played *that match*, as a plain integer string — this alone covers the "appearance weighting" need, no separate data source required.
- `understatapi`'s `base.py` has **no built-in rate-limiting/delay** between requests — a real acquisition script pulling hundreds of unique players needs to add its own politeness delay; this API is not designed to be hammered.

In [ ]:
try:
    c.player(player='999999999').get_match_data()
except InvalidPlayer as e:
    print('raised InvalidPlayer cleanly:', e)

sample = matches[0]
print()
print('field types (all strings):', {k: type(v).__name__ for k, v in list(sample.items())[:6]})

## Findings summary

See `README.md`.